# **02 컨텍스트 증강 및 프롬프트 엔지니어링**

### 학습 내용
1. 외부 정보(컨텍스트)를 프롬프트에 주입하기
2. 텍스트 파일을 활용한 정보 제공
3. 프롬프트 템플릿 활용
4. RAG의 기본 원리 이해

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("API Key가 설정되었습니다.")

API Key가 설정되었습니다.


## 1. LLM 초기화

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5.4-mini")

## 2. 외부 정보 없이 질문하기

먼저 LLM이 알지 못하는 특정 정보에 대해 질문해봅시다.

In [3]:
from IPython.display import Markdown, display

# LLM이 모를 가능성이 높은 질문
question = "우리 회사의 여름 휴가 정책이 어떻게 되나요?"

response = llm.invoke(question)
display(Markdown(response.content))

회사마다 여름 휴가 정책이 달라서, **당신 회사의 실제 정책은 내부 규정이나 인사팀 안내를 확인해야** 해요.  
제가 지금은 회사 내부 문서를 직접 볼 수 없어서 일반적인 경우만 안내드릴게요.

보통 여름 휴가 정책은 이런 항목으로 정리됩니다:
- **연차 사용 방식**: 여름 휴가가 별도 유급휴가인지, 연차에서 차감되는지
- **사용 가능 기간**: 예) 7~8월, 특정 성수기 기간
- **일수 제한**: 예) 2일, 3일, 5일 등
- **신청 절차**: 사전 승인 필요 여부, 신청 기한
- **동시 사용 제한**: 팀별 인원 분산을 위해 같은 기간에 최대 몇 명까지 가능한지
- **특별 지원**: 교통비/휴가비 지원 여부, 유급 보너스 휴가 등

원하시면 제가 바로 도와드릴 수 있어요:
1. **회사 공지/사내 규정 문구를 붙여주시면 해석**해드리기  
2. **인사팀에 물어볼 메일/메신저 문구**를 작성해드리기  
3. **여름 휴가 정책 확인 체크리스트**를 만들어드리기

원하시면 “인사팀에 물어볼 문구”로 바로 작성해드릴게요.

LLM은 학습 데이터에 없는 특정 정보(회사 내부 정책, 개인 정보 등)는 답변할 수 없습니다.

이를 해결하기 위해 **외부 정보를 프롬프트에 포함**시킬 수 있습니다.

## 3. 컨텍스트를 직접 추가하여 질문하기

필요한 정보를 프롬프트에 직접 포함시켜 봅시다.

In [4]:
# 회사 정책 정보 (컨텍스트)
context = """
우리 회사 여름 휴가 정책:
- 전 직원은 7월~8월 중 연속 5일의 여름 휴가를 사용할 수 있습니다.
- 휴가 신청은 최소 2주 전에 해야 합니다.
- 부서별로 최소 인원이 유지되어야 하므로 팀장과 사전 협의가 필요합니다.
- 여름 휴가는 연차와 별도로 제공되는 특별 휴가입니다.
"""

# 컨텍스트와 질문을 함께 전달
prompt = f"""
다음 정보를 참고하여 질문에 답변하세요.

[참고 정보]
{context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

우리 회사의 여름 휴가 정책은 다음과 같습니다.

- **대상:** 전 직원
- **사용 기간:** **7월~8월 중**
- **휴가 일수:** **연속 5일**
- **신청 시기:** **최소 2주 전**에 신청해야 함
- **사전 협의:** 부서별 최소 인원 유지를 위해 **팀장과 사전 협의 필요**
- **휴가 성격:** **연차와 별도로 제공되는 특별 휴가**

원하시면 제가 이 내용을 **사내 공지문 형태**로도 정리해드릴게요.

## 4. 텍스트 파일로부터 정보 읽어오기

실제 상황에서는 정보가 파일, 데이터베이스, 웹 페이지 등에 저장되어 있습니다.

텍스트 파일에서 정보를 읽어와서 프롬프트에 주입해봅시다.

In [4]:
# 먼저 샘플 텍스트 파일을 생성합니다
sample_text = """
상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템
"""

# 파일 저장
with open("course_info.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("샘플 텍스트 파일이 생성되었습니다: course_info.txt")

샘플 텍스트 파일이 생성되었습니다: course_info.txt


In [5]:
# 텍스트 파일 읽기
with open("course_info.txt", "r", encoding="utf-8") as f:
    course_context = f.read()

print("파일 내용:")
print(course_context)

파일 내용:

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템



In [7]:
# 파일에서 읽은 정보를 활용하여 질문하기
question = "이 과정의 학습 기간과 주요 학습 내용을 요약해주세요."

prompt = f"""
다음 과정 안내 정보를 참고하여 질문에 답변하세요.

[과정 정보]
{course_context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

이 과정은 **2026.08.18부터 2026.08.31까지 10일간, 총 80시간** 진행됩니다.  
주요 학습 내용은 다음과 같습니다.

- **LLM 애플리케이션 개발 기초**
- **RAG(검색증강생성) 시스템 구축**
- **Text2SQL 기반 데이터 조회 자동화**
- **AI Agent 시스템 개발**
- **LangGraph를 활용한 워크플로우 구성**

즉, **LLM 기반 서비스 개발의 기초부터 RAG, Text2SQL, AI Agent, LangGraph 워크플로우까지 실무 중심으로 학습하는 과정**입니다.

## 5. 프롬프트 템플릿 활용하기

LangChain의 `PromptTemplate`을 사용하면 프롬프트를 더 체계적으로 관리할 수 있습니다.

In [8]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 정의
template = """
당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 템플릿에 값 채우기
formatted_prompt = prompt_template.format(
    context=course_context,
    question="이 과정에서 어떤 포트폴리오를 완성하나요?"
)

print("생성된 프롬프트:")
print(formatted_prompt)
print("\n" + "="*80 + "\n")

response = llm.invoke(formatted_prompt)
print("답변:")
display(Markdown(response.content))

생성된 프롬프트:

당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템


[질문]
이 과정에서 어떤 포트폴리오를 완성하나요?

[답변]



답변:


이 과정의 최종 포트폴리오는 **2가지**입니다.

1. **RAG · Text2SQL 기반 데이터 조회 시스템**
2. **Tool 기반 AI Agent 시스템**

즉, 검색증강생성(RAG)과 Text2SQL을 활용한 데이터 조회 시스템, 그리고 도구를 활용하는 AI Agent 시스템을 완성하게 됩니다.

## 6. RAG의 기본 원리 이해

지금까지 실습한 내용이 바로 **RAG(Retrieval-Augmented Generation)** 의 핵심 원리입니다.

### RAG의 기본 흐름

1. **사용자 질문 입력**
2. **관련 문서 검색** (Retrieval)
   - 벡터 데이터베이스, 키워드 검색, 하이브리드 검색 등
3. **검색된 문서를 프롬프트에 주입** (Augmentation)
4. **LLM이 컨텍스트를 바탕으로 답변 생성** (Generation)

현재까지는 문서 검색 없이 직접 컨텍스트를 제공했지만,
다음 실습에서는 **벡터 데이터베이스를 활용한 자동 문서 검색**을 구현합니다.

## 📖 과제 1: 나만의 지식 베이스 만들기

자신이 관심 있는 주제나 전공 분야의 정보를 담은 텍스트 파일을 만들고,
해당 정보를 활용하여 질문-답변 시스템을 구현해보세요.

**예시 주제:**
- 좋아하는 영화/드라마의 줄거리와 등장인물 정보
- 자신의 포트폴리오나 이력서 내용
- 관심 분야의 용어 사전
- 수업 노트나 요약 자료

**구현 요구사항:**
1. 텍스트 파일(.txt) 생성 (최소 200자 이상)
2. 파일 내용을 읽어와서 컨텍스트로 활용
3. 3개 이상의 질문을 만들어 답변 생성

In [6]:
# TODO 1. 나만의 지식 베이스 내용 작성 (200자 이상)
my_knowledge_base = """
[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.
"""

# TODO 2. 파일명 설정
filename = "my_stock_market_info.txt"

# TODO 3. 질문 3개 작성
questions = [
    "AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?",
    "기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.",
    "기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?"
]

# 텍스트 파일 생성
with open(filename, "w", encoding="utf-8") as f:
    f.write(my_knowledge_base)
print(f"✓ 파일이 생성되었습니다: {filename}\n")

# 파일 내용 읽기
with open(filename, "r", encoding="utf-8") as f:
    context = f.read()

# 프롬프트 템플릿 정의
from langchain_core.prompts import PromptTemplate

template = """
당신은 도움이 되는 금융 및 증권 전문 AI 어시스턴트입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 각 질문에 대해 답변 생성
from IPython.display import Markdown, display

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"질문 {i}: {question}")
    print('='*80)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    print("생성된 프롬프트:")
    print(formatted_prompt)
    print("\n" + "="*80 + "\n")

    response = llm.invoke(formatted_prompt)
    print("답변:")
    display(Markdown(response.content))

✓ 파일이 생성되었습니다: my_stock_market_info.txt


질문 1: AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?
생성된 프롬프트:

당신은 도움이 되는 금융 및 증권 전문 AI 어시스턴트입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?

[답변]



답변:


AI 설비투자 확대와 전력 수요 급증으로 수혜를 받고 있는 주요 섹터는 다음과 같습니다.

1. **AI 반도체 및 인프라 섹터**
   - **HBM(고대역폭 메모리)** 관련 기업
   - **차세대 패키징 장비** 기업

2. **전력 인프라 섹터**
   - **변압기**
   - **전선** 관련 기업

즉, 글로벌 빅테크의 AI CAPEX 확대는 **AI 반도체·인프라**를, 전력 수요 급증은 **전력 인프라** 섹터를 중심으로 수혜를 주고 있습니다.


질문 2: 기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.
생성된 프롬프트:

당신은 도움이 되는 금융 및 증권 전문 AI 어시스턴트입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.

[답변]



답변:


기업 밸류업 프로그램에서 **주주환원 정책을 적극적으로 펼치는 주요 업종**은 참고 정보상 다음과 같습니다.

- **지주사**
- **금융**
- **자동차**

이들 업종은 대체로 **저PBR 종목군**에 속하며, 최근에는 **자사주 소각**과 **배당 확대** 같은 주주환원 정책 공시가 확대되고 있습니다.

### 시장 반응
- **주주환원율이 높고 자본 효율성이 우수한 기업**을 담은 **밸류업 지수**와 관련 **ETF로 자금 유입이 지속**되고 있습니다.
- 즉, 시장은 이런 기업들의 주주환원 강화에 대해 **긍정적으로 반응**하고 있으며, 관련 종목과 상품에 대한 투자 수요가 늘고 있는 흐름입니다.

제공된 정보 범위에서는 이 외의 세부 업종이나 개별 기업 사례는 확인할 수 없습니다.


질문 3: 기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?
생성된 프롬프트:

당신은 도움이 되는 금융 및 증권 전문 AI 어시스턴트입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?

[답변]



답변:


기준금리 인하 기조는 일반적으로 **성장주에 긍정적**입니다.

이유는 다음과 같습니다.

1. **할인율 하락으로 미래 이익의 현재가치가 높아짐**
   - 성장주는 현재 이익보다 **미래의 높은 성장성**을 더 크게 평가받는 경우가 많습니다.
   - 기준금리가 내려가면 채권 금리도 하향 안정화되는 경향이 있어, 미래 현금흐름을 할인하는 기준이 낮아집니다.
   - 그 결과 **성장주의 밸류에이션 부담이 완화**됩니다.

2. **자금 조달 비용 감소**
   - 금리가 낮아지면 기업과 시장 전반의 자금 조달 비용이 줄어들 수 있습니다.
   - 특히 바이오, 소프트웨어처럼 연구개발이나 투자 비중이 큰 성장기업에는 우호적입니다.

3. **위험자산 선호 확대 가능성**
   - 금리 인하 기대가 커지면 예금·채권보다 주식, 특히 성장주 같은 위험자산으로 자금이 이동할 수 있습니다.

정리하면, 제공된 정보 기준으로 **기준금리 인하 기대는 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담을 낮추고 투자심리를 개선하는 요인**으로 볼 수 있습니다.

## 📖 과제 2: 프롬프트 최적화하기

같은 컨텍스트와 질문이라도 프롬프트를 어떻게 구성하느냐에 따라 답변 품질이 달라집니다.

다음 요소들을 추가하여 프롬프트를 개선해보세요:

1. **역할 정의**: "당신은 ~한 전문가입니다"
2. **답변 형식 지정**: "다음 형식으로 답변하세요: ..."
3. **제약 조건**: "정보에 없는 내용은 '정보 없음'이라고 답하세요"
4. **예시 제공**: Few-shot learning (예시 포함)

원본 프롬프트와 개선된 프롬프트의 답변을 비교해보세요.

In [7]:
# TODO 1. 나만의 지식 베이스 내용 작성 (200자 이상)
my_knowledge_base = """
[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.
"""

# TODO 2. 파일명 설정
filename = "my_stock_market_info.txt"

# TODO 3. 질문 3개 작성
questions = [
    "AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?",
    "기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.",
    "기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?"
]

# 텍스트 파일 생성
with open(filename, "w", encoding="utf-8") as f:
    f.write(my_knowledge_base)
print(f"✓ 파일이 생성되었습니다: {filename}\n")

# 파일 내용 읽기
with open(filename, "r", encoding="utf-8") as f:
    context = f.read()

# 프롬프트 템플릿 정의
from langchain_core.prompts import PromptTemplate

template = """
당신은 금융 전문가입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.
예시도 작성해주세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 각 질문에 대해 답변 생성
from IPython.display import Markdown, display

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"질문 {i}: {question}")
    print('='*80)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    print("생성된 프롬프트:")
    print(formatted_prompt)
    print("\n" + "="*80 + "\n")

    response = llm.invoke(formatted_prompt)
    print("답변:")
    display(Markdown(response.content))

✓ 파일이 생성되었습니다: my_stock_market_info.txt


질문 1: AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?
생성된 프롬프트:

당신은 금융 전문가입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.
예시도 작성해주세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
AI 설비투자 확대와 전력 수요 급증으로 인해 수혜를 받고 있는 주요 섹터는 무엇인가요?

[답변]



답변:


AI 설비투자 확대와 전력 수요 급증으로 수혜를 받고 있는 주요 섹터는 **AI 반도체 및 인프라 섹터**와 **전력 인프라 섹터**입니다.

- **AI 반도체 및 인프라 섹터**:  
  글로벌 빅테크의 AI CAPEX 확대에 따라 **HBM(고대역폭 메모리)**, **차세대 패키징 장비** 관련 기업들의 실적이 호조를 보이고 있습니다.

- **전력 인프라 섹터**:  
  전력 수요가 급증하면서 **변압기**, **전선** 등 전력 인프라 관련 종목군이 강한 실적 모멘텀을 유지하고 있습니다.

### 예시
- AI 서버 수요 증가로 **HBM 생산 기업**의 실적이 개선될 수 있습니다.
- 데이터센터 전력 사용량 확대에 따라 **변압기 제조업체**와 **전선 업체**가 수혜를 받을 수 있습니다.


질문 2: 기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.
생성된 프롬프트:

당신은 금융 전문가입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.
예시도 작성해주세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
기업 밸류업 프로그램에서 주주환원 정책을 적극적으로 펼치는 주요 업종과 시장 반응을 알려주세요.

[답변]



답변:


기업 밸류업 프로그램에서 **주주환원 정책을 적극적으로 펼치는 주요 업종**은 주로 다음과 같습니다.

### 주요 업종
1. **지주사**
   - 상대적으로 저PBR인 경우가 많고, 자사주 소각이나 배당 확대를 통한 주주환원 강화가 자주 나타납니다.

2. **금융업**
   - 은행, 보험 등은 안정적인 현금흐름을 바탕으로 배당 확대와 자사주 매입·소각을 추진하는 사례가 많습니다.

3. **자동차 업종**
   - 수익성이 개선된 기업을 중심으로 배당 확대, 자사주 소각 등 주주환원 정책이 확대되는 흐름이 있습니다.

### 시장 반응
- **밸류업 지수와 관련 ETF로의 자금 유입이 지속**되고 있습니다.
- 특히 **주주환원율이 높고 자본 효율성이 우수한 기업**이 포함된 종목군에 대한 투자 선호가 높아지고 있습니다.
- 결과적으로 **저PBR 업종 중심의 재평가 기대감**이 커지고 있습니다.

### 예시
- 어떤 **금융지주사**가 배당을 늘리고 자사주를 소각한다고 공시하면, 시장에서는 이를 긍정적으로 해석해 해당 종목과 밸류업 관련 ETF에 자금이 유입될 수 있습니다.
- **자동차 기업**이 적극적인 배당 정책을 발표하면, 저평가 해소 기대가 반영되어 주가가 강세를 보일 수 있습니다.

추가로 원하시면 제가 이 내용을 **한 줄 요약**이나 **표 형태**로도 정리해드릴 수 있습니다.


질문 3: 기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?
생성된 프롬프트:

당신은 금융 전문가입니다.
주어진 [참고 정보]를 바탕으로 사용자의 질문에 명확하고 정확하게 답변하세요.
만약 주어진 정보로 확인할 수 없는 질문이라면 솔직하게 '제공된 정보에서는 확인할 수 없습니다'라고 밝히세요.
예시도 작성해주세요.

[참고 정보]

[2026년 하반기 국내 및 글로벌 증권 시장 동향 요약]

1. AI 반도체 및 인프라 섹터 동향:
글로벌 빅테크의 AI 설비투자(CAPEX) 확대 지속에 따라 고대역폭 메모리(HBM) 및 차세대 패키징 장비 기업들의 실적 호조가 이어지고 있습니다. 전력 수요 급증에 따른 변압기, 전선 등 전력 인프라 관련 종목군 역시 강한 실적 모멘텀을 유지하고 있습니다.

2. 기업 밸류업 프로그램 및 주주환원:
국내 증시에서는 저PBR 종목(지주사, 금융, 자동차 등)을 중심으로 자사주 소각 및 배당 확대 등 주주환원 정책 공시가 확대되고 있습니다. 주주환원율이 높고 자본 효율성이 우수한 기업을 편입한 밸류업 지수 및 관련 ETF로의 자금 유입이 지속되는 추세입니다.

3. 통화 정책 및 채권/증시 영향:
주요국 중앙은행의 기준금리 인하 기조 전환 기대감으로 국채 금리가 하향 안정화되는 흐름을 보이고 있으며, 이에 따라 성장주(바이오, 소프트웨어 등)의 밸류에이션 부담이 완화되는 효과가 나타나고 있습니다.


[질문]
기준금리 인하 기조가 성장주에 미치는 영향은 무엇인가요?

[답변]



답변:


기준금리 인하 기조는 **성장주(예: 바이오, 소프트웨어 등)**에 대체로 **긍정적**입니다.  
제공된 정보에 따르면, 중앙은행의 기준금리 인하 기대감으로 **국채 금리가 하향 안정화**되면 성장주의 미래 이익을 현재가치로 할인하는 부담이 줄어들어 **밸류에이션 부담이 완화**되는 효과가 있습니다.

즉, 금리가 내려갈수록 시장은 성장주의 미래 성장성을 더 높게 평가하는 경향이 있어, 주가에 우호적으로 작용할 수 있습니다.

### 예시
- **예시 1:** 바이오 기업처럼 현재 이익은 적지만 미래 성장성이 큰 회사는, 금리가 낮아지면 미래 수익의 가치가 더 크게 평가되어 주가에 긍정적 영향을 받을 수 있습니다.
- **예시 2:** 소프트웨어 기업은 장기 성장 기대가 중요한데, 금리 인하로 할인율이 낮아지면 예상 현금흐름의 현재가치가 올라가 밸류에이션이 개선될 수 있습니다.

다만, 개별 종목의 실적이나 산업 경쟁 상황에 따라 실제 주가 반응은 달라질 수 있습니다.

---

### 참고 자료

- [LangChain Prompts 공식 문서](https://python.langchain.com/docs/modules/model_io/prompts/)
- [프롬프트 엔지니어링 가이드](https://www.promptingguide.ai/)